In [1]:
!pip install mysql-connector-python

In [13]:
import mysql.connector

In [14]:
mydb = mysql.connector.connect(
    host = "localhost",
    user = "root",
    password="",
    port = "3306"
)

In [15]:
mycursor = mydb.cursor(buffered=True)

In [16]:
mycursor.execute("SHOW DATABASES")

for x in mycursor:
    print(x)


('guvi',)
('information_schema',)
('mysql',)
('performance_schema',)
('phpmyadmin',)
('project_i',)
('test',)


In [17]:
mycursor.execute("CREATE DATABASE Project_I")

DatabaseError: 1007 (HY000): Can't create database 'project_i'; database exists

In [18]:
mycursor.execute("USE Project_I")

In [19]:
from tabulate import tabulate

In [12]:
import pandas as pd
import mysql.connector

# Step 1: Read CSV into DataFrame
df = pd.read_csv("Guvi/Projects/Uber_Eats/uber_29.4.2026.csv")

# Step 2: Connect to MySQL
mydb = mysql.connector.connect(
    host = "localhost",
    user = "root",
    password="",
    port = "3306",
    database="project_i"
)

mycursor = mydb.cursor()

# Step 3: Create table (SQL inside triple quotes)
mycursor.execute("""
CREATE TABLE IF NOT EXISTS Uber_explore (
    name VARCHAR(50),
    online_order INT,
    book_table INT,
    rate FLOAT,
    votes INT,
    phone VARCHAR(50),
    location VARCHAR(50),
    res_type VARCHAR(50),
    dish_liked VARCHAR(50),
    cuisines VARCHAR(50),
    approx_cost_for_two_people VARCHAR(50),
    listed_in_type VARCHAR(50),
    listed_in_city VARCHAR(50),
    rest_type_primary VARCHAR(50)
)
""")

# Step 4: Insert rows from DataFrame
for _, row in df.iterrows():
    mycursor.execute("""
        INSERT INTO Uber_explore 
        (name, online_order, book_table, rate, votes, phone, location, res_type, dish_liked, cuisines, approx_cost_for_two_people, listed_in_type, listed_in_city, rest_type_primary)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """, (
        row['name'], row['online_order'], row['book_table'], row['rate'], row['votes'],
        row['phone'], row['location'], row['rest_type'], row['dish_liked'], row['cuisines'],
        row['approx_cost_for_two_people'], row['listed_in_type'], row['listed_in_city'], row['rest_type_primary']
    ))

# Step 5: Commit and close
mydb.commit()
mycursor.close()
mydb.close()

print("Data inserted successfully!")

Data inserted successfully!


In [21]:
mycursor.execute("""
SELECT COUNT(*) AS total_rows
FROM uber_explore;
""")
out = mycursor.fetchall()
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))

+--------------+
|   total_rows |
|--------------|
|        23012 |
+--------------+


In [22]:
mycursor.execute("""
SELECT COUNT(*) AS total_columns
FROM INFORMATION_SCHEMA.COLUMNS
WHERE table_schema = DATABASE()
  AND table_name = 'uber_explore';
""")
out = mycursor.fetchall()
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))

+-----------------+
|   total_columns |
|-----------------|
|              14 |
+-----------------+


In [25]:
mycursor.execute("DESCRIBE uber_explore")
out = mycursor.fetchall()
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))

+----------------------------+-------------+--------+-------+-----------+---------+
| Field                      | Type        | Null   | Key   | Default   | Extra   |
|----------------------------+-------------+--------+-------+-----------+---------|
| name                       | varchar(50) | YES    |       |           |         |
| online_order               | int(11)     | YES    |       |           |         |
| book_table                 | int(11)     | YES    |       |           |         |
| rate                       | float       | YES    |       |           |         |
| votes                      | int(11)     | YES    |       |           |         |
| phone                      | varchar(50) | YES    |       |           |         |
| location                   | varchar(50) | YES    |       |           |         |
| res_type                   | varchar(50) | YES    |       |           |         |
| dish_liked                 | varchar(50) | YES    |       |           |   

In [26]:
mycursor.execute("SHOW COLUMNS FROM uber_explore")
out = mycursor.fetchall()
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))

+----------------------------+-------------+--------+-------+-----------+---------+
| Field                      | Type        | Null   | Key   | Default   | Extra   |
|----------------------------+-------------+--------+-------+-----------+---------|
| name                       | varchar(50) | YES    |       |           |         |
| online_order               | int(11)     | YES    |       |           |         |
| book_table                 | int(11)     | YES    |       |           |         |
| rate                       | float       | YES    |       |           |         |
| votes                      | int(11)     | YES    |       |           |         |
| phone                      | varchar(50) | YES    |       |           |         |
| location                   | varchar(50) | YES    |       |           |         |
| res_type                   | varchar(50) | YES    |       |           |         |
| dish_liked                 | varchar(50) | YES    |       |           |   

In [28]:
mycursor.execute("""
SELECT 
    SUM(CASE WHEN name IS NULL THEN 1 ELSE 0 END) AS name_nulls,
    SUM(CASE WHEN online_order IS NULL THEN 1 ELSE 0 END) AS online_order_nulls,
    SUM(CASE WHEN book_table IS NULL THEN 1 ELSE 0 END) AS book_table_nulls,
    SUM(CASE WHEN rate IS NULL THEN 1 ELSE 0 END) AS rate_nulls,
    SUM(CASE WHEN votes IS NULL THEN 1 ELSE 0 END) AS votes_nulls,
    SUM(CASE WHEN phone IS NULL THEN 1 ELSE 0 END) AS phone_nulls,
    SUM(CASE WHEN location IS NULL THEN 1 ELSE 0 END) AS location_nulls,
    SUM(CASE WHEN res_type IS NULL THEN 1 ELSE 0 END) AS res_type_nulls,
    SUM(CASE WHEN dish_liked IS NULL THEN 1 ELSE 0 END) AS dish_liked_nulls,
    SUM(CASE WHEN cuisines IS NULL THEN 1 ELSE 0 END) AS cuisines_nulls,
    SUM(CASE WHEN approx_cost_for_two_people IS NULL THEN 1 ELSE 0 END) AS cost_nulls,
    SUM(CASE WHEN listed_in_type IS NULL THEN 1 ELSE 0 END) AS listed_type_nulls,
    SUM(CASE WHEN listed_in_city IS NULL THEN 1 ELSE 0 END) AS listed_city_nulls,
    SUM(CASE WHEN rest_type_primary IS NULL THEN 1 ELSE 0 END) AS rest_type_primary_nulls
FROM uber_explore;
""")
out = mycursor.fetchall()
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))

+--------------+----------------------+--------------------+--------------+---------------+---------------+------------------+------------------+--------------------+------------------+--------------+---------------------+---------------------+---------------------------+
|   name_nulls |   online_order_nulls |   book_table_nulls |   rate_nulls |   votes_nulls |   phone_nulls |   location_nulls |   res_type_nulls |   dish_liked_nulls |   cuisines_nulls |   cost_nulls |   listed_type_nulls |   listed_city_nulls |   rest_type_primary_nulls |
|--------------+----------------------+--------------------+--------------+---------------+---------------+------------------+------------------+--------------------+------------------+--------------+---------------------+---------------------+---------------------------|
|            0 |                    0 |                  0 |            0 |             0 |             0 |                0 |                0 |                  0 |               

In [29]:
mycursor.execute("""
SELECT *
FROM uber_explore
WHERE name IS NULL
   OR online_order IS NULL
   OR book_table IS NULL
   OR rate IS NULL
   OR votes IS NULL
   OR phone IS NULL
   OR location IS NULL
   OR res_type IS NULL
   OR dish_liked IS NULL
   OR cuisines IS NULL
   OR approx_cost_for_two_people IS NULL
   OR listed_in_type IS NULL
   OR listed_in_city IS NULL
   OR rest_type_primary IS NULL;
""")
out = mycursor.fetchall()
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))

+--------+----------------+--------------+--------+---------+---------+------------+------------+--------------+------------+------------------------------+------------------+------------------+---------------------+
| name   | online_order   | book_table   | rate   | votes   | phone   | location   | res_type   | dish_liked   | cuisines   | approx_cost_for_two_people   | listed_in_type   | listed_in_city   | rest_type_primary   |
|--------+----------------+--------------+--------+---------+---------+------------+------------+--------------+------------+------------------------------+------------------+------------------+---------------------|
+--------+----------------+--------------+--------+---------+---------+------------+------------+--------------+------------+------------------------------+------------------+------------------+---------------------+


In [30]:
#1.Locations with Highest Average Restaurant Ratings
mycursor.execute("""
SELECT location,
       ROUND(AVG(rate),2) AS avg_rating,
       COUNT(*) AS restaurant_count
FROM uber_explore
GROUP BY location
HAVING COUNT(*) >= 10
ORDER BY avg_rating DESC;
""")
out = mycursor.fetchall()
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))

+-------------------------------+--------------+--------------------+
| location                      |   avg_rating |   restaurant_count |
|-------------------------------+--------------+--------------------|
| Lavelle Road                  |         4.19 |                442 |
| Koramangala 5Th Block         |         4.15 |               1759 |
| Sankey Road                   |         4.11 |                 17 |
| Koramangala 3Rd Block         |         4.1  |                161 |
| Cunningham Road               |         4.1  |                333 |
| St. Marks Road                |         4.1  |                304 |
| Koramangala 2Nd Block         |         4.07 |                 45 |
| Sadashiv Nagar                |         4.06 |                 38 |
| Residency Road                |         4.05 |                440 |
| Church Street                 |         4.05 |                506 |
| Koramangala 4Th Block         |         4.04 |                632 |
| Seshadripuram     

In [31]:
#2️. Over-Saturated Locations (Most Restaurants)
mycursor.execute("""
SELECT location,
       COUNT(*) AS restaurant_count
FROM uber_explore
GROUP BY location
ORDER BY restaurant_count DESC;
""")
out = mycursor.fetchall()
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))

+-------------------------------+--------------------+
| location                      |   restaurant_count |
|-------------------------------+--------------------|
| Koramangala 5Th Block         |               1759 |
| Btm                           |               1444 |
| Indiranagar                   |               1331 |
| Hsr                           |               1154 |
| Jayanagar                     |               1030 |
| Jp Nagar                      |                989 |
| Whitefield                    |                821 |
| Koramangala 6Th Block         |                718 |
| Koramangala 7Th Block         |                715 |
| Marathahalli                  |                678 |
| Koramangala 4Th Block         |                632 |
| Mg Road                       |                589 |
| Brigade Road                  |                565 |
| Church Street                 |                506 |
| Bannerghatta Road             |                496 |
| Ulsoor  

In [33]:
#3.Does Online Ordering Improve Ratings?
mycursor.execute("""
SELECT online_order,
       ROUND(AVG(rate),2) AS avg_rating,
       COUNT(*) AS restaurant_count
FROM uber_explore
GROUP BY online_order;
""")
out = mycursor.fetchall()
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))

+----------------+--------------+--------------------+
|   online_order |   avg_rating |   restaurant_count |
|----------------+--------------+--------------------|
|              0 |         3.93 |               6738 |
|              1 |         3.89 |              16274 |
+----------------+--------------+--------------------+


In [34]:
#4.Does Table Booking Improve Ratings?
mycursor.execute("""
SELECT book_table,
       ROUND(AVG(rate),2) AS avg_rating,
       COUNT(*) AS restaurant_count
FROM uber_explore
GROUP BY book_table;
""")
out = mycursor.fetchall()
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))

+--------------+--------------+--------------------+
|   book_table |   avg_rating |   restaurant_count |
|--------------+--------------+--------------------|
|            0 |         3.81 |              16991 |
|            1 |         4.16 |               6021 |
+--------------+--------------+--------------------+


In [35]:
#5. Best Price Range for Customer Satisfaction
mycursor.execute("""
SELECT
  CASE
    WHEN CAST(approx_cost_for_two_people AS UNSIGNED) < 300 THEN 'Low'
    WHEN CAST(approx_cost_for_two_people AS UNSIGNED) BETWEEN 300 AND 700 THEN 'Mid'
    ELSE 'Premium'
  END AS price_range,
  ROUND(AVG(rate),2) AS avg_rating
FROM uber_explore
GROUP BY price_range;
""")
out = mycursor.fetchall()
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))

+---------------+--------------+
| price_range   |   avg_rating |
|---------------+--------------|
| Low           |         3.84 |
| Mid           |         3.79 |
| Premium       |         4.07 |
+---------------+--------------+


In [43]:
mycursor.execute("""
DESCRIBE uber_explore;
""")
out = mycursor.fetchall()
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt="psql"))

+----------------------------+-------------+--------+-------+-----------+---------+
| Field                      | Type        | Null   | Key   | Default   | Extra   |
|----------------------------+-------------+--------+-------+-----------+---------|
| name                       | varchar(50) | YES    |       |           |         |
| online_order               | int(11)     | YES    |       |           |         |
| book_table                 | int(11)     | YES    |       |           |         |
| rate                       | float       | YES    |       |           |         |
| votes                      | int(11)     | YES    |       |           |         |
| phone                      | varchar(50) | YES    |       |           |         |
| location                   | varchar(50) | YES    |       |           |         |
| res_type                   | varchar(50) | YES    |       |           |         |
| dish_liked                 | varchar(50) | YES    |       |           |   

In [36]:
#10. Cost vs Rating Relationship
mycursor.execute("""
SELECT CAST(approx_cost_for_two_people AS UNSIGNED) AS cost,
       ROUND(AVG(rate),2) AS avg_rating
FROM uber_explore
GROUP BY cost
ORDER BY cost;
""")
out = mycursor.fetchall()
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))

+--------+--------------+
|   cost |   avg_rating |
|--------+--------------|
|     40 |         3.73 |
|    100 |         3.97 |
|    120 |         4.4  |
|    150 |         3.95 |
|    180 |         4.22 |
|    200 |         3.84 |
|    230 |         3.99 |
|    250 |         3.75 |
|    300 |         3.82 |
|    330 |         2.7  |
|    350 |         3.71 |
|    400 |         3.82 |
|    450 |         3.68 |
|    500 |         3.75 |
|    550 |         3.81 |
|    600 |         3.82 |
|    650 |         3.72 |
|    700 |         3.87 |
|    750 |         3.89 |
|    800 |         3.93 |
|    850 |         3.89 |
|    900 |         4.06 |
|    950 |         3.88 |
|   1000 |         4    |
|   1050 |         2.57 |
|   1100 |         4.09 |
|   1200 |         4.09 |
|   1250 |         4.32 |
|   1300 |         4.18 |
|   1350 |         3.78 |
|   1400 |         4.24 |
|   1450 |         4.02 |
|   1500 |         4.2  |
|   1600 |         4.34 |
|   1650 |         4.3  |
|   1700 |  

In [38]:
#6. Low / Mid / Premium Performance Comparison
mycursor.execute("""
SELECT
  CASE
    WHEN CAST(approx_cost_for_two_people AS UNSIGNED) < 300 THEN 'Low'
    WHEN CAST(approx_cost_for_two_people AS UNSIGNED) BETWEEN 300 AND 700 THEN 'Mid'
    ELSE 'Premium'
  END AS price_segment,
  COUNT(*) AS restaurant_count,
  ROUND(AVG(rate),2) AS avg_rating
FROM uber_explore
GROUP BY price_segment;
""")
out = mycursor.fetchall()
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))

+-----------------+--------------------+--------------+
| price_segment   |   restaurant_count |   avg_rating |
|-----------------+--------------------+--------------|
| Low             |               2069 |         3.84 |
| Mid             |              12073 |         3.79 |
| Premium         |               8870 |         4.07 |
+-----------------+--------------------+--------------+


In [39]:
#12.High Demand but Low Ratings
mycursor.execute("""
SELECT location,
       COUNT(*) AS restaurant_count,
       ROUND(AVG(rate),2) AS avg_rating
FROM uber_explore
GROUP BY location
HAVING restaurant_count > 50 AND avg_rating < 4.0
ORDER BY restaurant_count DESC;
""")
out = mycursor.fetchall()
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))


+-----------------------+--------------------+--------------+
| location              |   restaurant_count |   avg_rating |
|-----------------------+--------------------+--------------|
| Btm                   |               1444 |         3.76 |
| Indiranagar           |               1331 |         3.96 |
| Hsr                   |               1154 |         3.84 |
| Jayanagar             |               1030 |         3.94 |
| Jp Nagar              |                989 |         3.85 |
| Whitefield            |                821 |         3.82 |
| Koramangala 6Th Block |                718 |         3.95 |
| Koramangala 7Th Block |                715 |         3.99 |
| Marathahalli          |                678 |         3.73 |
| Mg Road               |                589 |         3.95 |
| Brigade Road          |                565 |         3.94 |
| Bannerghatta Road     |                496 |         3.68 |
| Koramangala 1St Block |                466 |         3.87 |
| Kalyan

In [40]:
#13.Online Order + Table Booking Performance
mycursor.execute("""
SELECT online_order,
       book_table,
       ROUND(AVG(rate),2) AS avg_rating,
       COUNT(*) AS restaurant_count
FROM uber_explore
GROUP BY online_order, book_table;
""")
out = mycursor.fetchall()
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))


+----------------+--------------+--------------+--------------------+
|   online_order |   book_table |   avg_rating |   restaurant_count |
|----------------+--------------+--------------+--------------------|
|              0 |            0 |         3.79 |               4331 |
|              0 |            1 |         4.18 |               2407 |
|              1 |            0 |         3.82 |              12660 |
|              1 |            1 |         4.15 |               3614 |
+----------------+--------------+--------------+--------------------+


In [80]:
#.15.Top Performers Within Each Pricing Segment
mycursor.execute("""
SELECT name,
       rate,
       CAST(approx_cost_for_two_people AS UNSIGNED) AS cost,
       CASE
         WHEN CAST(approx_cost_for_two_people AS UNSIGNED) < 300 THEN 'Low'
         WHEN CAST(approx_cost_for_two_people AS UNSIGNED) BETWEEN 300 AND 700 THEN 'Mid'
         ELSE 'Premium'
       END AS price_segment
FROM uber_explore
WHERE rate >= 4.5
ORDER BY price_segment, rate DESC;
""")


InternalError: Unread result found

In [48]:
import json
import mysql.connector

# Connect to MySQL
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="",        # XAMPP default
    database="project_i"   # change if needed
)

mycursor = conn.cursor()

# Create the orders table if it doesn't exist
create_table_sql = """
CREATE TABLE IF NOT EXISTS orders (
    id INT AUTO_INCREMENT PRIMARY KEY,
    order_id VARCHAR(255),
    restaurant_name VARCHAR(255),
    order_date DATE,
    order_value DECIMAL(10, 2),
    discount_used DECIMAL(10, 2),
    payment_method VARCHAR(100)
)
"""

# Execute table creation
mycursor.execute(create_table_sql)
print("Table 'orders' created or already exists.")

# Load orders.json
with open(r"D:\Guvi\Projects\Uber_Eats\orders.json", "r") as f:
    orders = json.load(f)
    
# SQL insert query
insert_sql = """
INSERT INTO orders
(order_id, restaurant_name, order_date, order_value, discount_used, payment_method)
VALUES (%s, %s, %s, %s, %s, %s)
"""

# Insert records
for o in orders:
    mycursor.execute(insert_sql, (
        o["order_id"],
        o["restaurant_name"],
        o["order_date"],
        o["order_value"],
        o["discount_used"],
        o["payment_method"]
    ))

# Commit changes
conn.commit()

# Close connection
mycursor.close()
conn.close()

print("orders.json imported successfully into MySQL!")

Table 'orders' created or already exists.
orders.json imported successfully into MySQL!


In [58]:
import json
import mysql.connector

# Connect to MySQL
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="",        # XAMPP default
    database="project_i"   # change if needed
)

mycursor = conn.cursor()

In [ ]:
#Which restaurants generate the highest total revenue?

In [64]:
#1 Total orders per restaurant
mycursor.execute("""
SELECT restaurant_name,
       COUNT(*) AS total_orders
FROM orders
GROUP BY restaurant_name
ORDER BY total_orders DESC LIMIT 10;
""")
out = mycursor.fetchall()
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))

+------------------------------------+----------------+
| restaurant_name                    |   total_orders |
|------------------------------------+----------------|
| Khan Saheb Grills and Rolls        |             25 |
| Kesar Sweet Shop and Fast Food     |             20 |
| Andhra Grills                      |             19 |
| Cake Cafe                          |             19 |
| KKR FOODIES                        |             19 |
| Orbis Restaurant                   |             18 |
| Hungry Lee                         |             18 |
| Reddy's Restaurant                 |             18 |
| Shree Muthahalli Veg               |             18 |
| JW Kitchen - JW Marriott Bengaluru |             17 |
+------------------------------------+----------------+


In [ ]:
#What is the average order value by payment method?

In [65]:
#2,Revenue generated by each restaurant
mycursor.execute("""
SELECT restaurant_name,
       ROUND(SUM(order_value),2) AS total_revenue
FROM orders
GROUP BY restaurant_name
ORDER BY total_revenue DESC LIMIT 15;
""")
out = mycursor.fetchall()
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))

+--------------------------------+-----------------+
| restaurant_name                |   total_revenue |
|--------------------------------+-----------------|
| Kesar Sweet Shop and Fast Food |         25222.4 |
| Khan Saheb Grills and Rolls    |         23507.2 |
| Bob's Bar                      |         19518.1 |
| Cake Cafe                      |         19335.2 |
| Biryani Mane                   |         19249.5 |
| Hungry Lee                     |         19167.5 |
| Andhra Grills                  |         19153.7 |
| Mighty Paws                    |         18410.5 |
| KKR FOODIES                    |         18137.2 |
| Delhi Ke Bawarchi              |         18131.7 |
| Pingara                        |         17954.1 |
| Bawarchi Inn                   |         17717.4 |
| Soup'ermanz Kitchen            |         17350.7 |
| Red Stone Kitchen              |         17333.8 |
| Carnatic                       |         17121   |
+--------------------------------+------------

In [ ]:
#Do discounts increase order value?

In [62]:
#3. Do high-rated restaurants get more orders?
mycursor.execute("""
SELECT u.rate,
       COUNT(o.order_id) AS total_orders
FROM uber_explore u
JOIN orders o
  ON u.name = o.restaurant_name
GROUP BY u.rate
ORDER BY u.rate DESC;
""")
out = mycursor.fetchall()
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))

+--------+----------------+
|   rate |   total_orders |
|--------+----------------|
|    4.9 |            432 |
|    4.8 |            538 |
|    4.7 |           1590 |
|    4.6 |           2314 |
|    4.5 |           5005 |
|    4.4 |           8581 |
|    4.3 |          13251 |
|    4.2 |          16900 |
|    4.1 |          22431 |
|    4   |          22911 |
|    3.9 |          25507 |
|    3.8 |          19604 |
|    3.7 |          12592 |
|    3.6 |           6739 |
|    3.5 |           3904 |
|    3.4 |           2919 |
|    3.3 |           1881 |
|    3.2 |           2108 |
|    3.1 |           2271 |
|    3   |           2220 |
|    2.9 |           2421 |
|    2.8 |           2350 |
|    2.7 |           1618 |
|    2.6 |           1253 |
|    2.5 |            489 |
|    2.4 |            366 |
|    2.3 |            337 |
|    2.2 |            282 |
|    2.1 |            167 |
|    2   |             66 |
|    1.8 |             65 |
+--------+----------------+


In [ ]:
#How does daily order revenue trend over time?

In [63]:
#4 Impact of discounts on order value
mycursor.execute("""
SELECT discount_used,
       ROUND(AVG(order_value),2) AS avg_order_value,
       COUNT(*) AS order_count
FROM orders
GROUP BY discount_used;
""")
out = mycursor.fetchall()
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))

+-----------------+-------------------+---------------+
|   discount_used |   avg_order_value |   order_count |
|-----------------+-------------------+---------------|
|               0 |            986.08 |         25000 |
+-----------------+-------------------+---------------+


In [ ]:
#5 Which are the top 10 restaurants with the highest average order value, considering only restaurants with at least 5 orders?

In [76]:
mycursor.execute("""
SELECT restaurant_name,
       ROUND(AVG(order_value), 2) AS avg_order_value,
       COUNT(order_id) AS total_orders
FROM orders
GROUP BY restaurant_name
HAVING COUNT(order_id) >= 5
ORDER BY avg_order_value DESC limit 10;
""")
out = mycursor.fetchall()
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))

+----------------------------------+-------------------+----------------+
| restaurant_name                  |   avg_order_value |   total_orders |
|----------------------------------+-------------------+----------------|
| Chatori Tongue                   |           1690.9  |              5 |
| Chow San                         |           1610.83 |              5 |
| Biting Junction                  |           1572.31 |              5 |
| Bikaner Jn                       |           1554.2  |              5 |
| Mockaholic Restro Beer Cafe      |           1545.11 |              5 |
| Momo Jojo                        |           1523.28 |              7 |
| Hotel Halli Nenapu               |           1512.58 |              6 |
| Burrito Boys                     |           1511.91 |             10 |
| Alba - JW Marriott Bengaluru     |           1511.48 |              5 |
| Angaar Indian Grill & Restaurant |           1495.73 |             10 |
+----------------------------------+--

In [ ]:
#6 Which payment methods generate the highest total revenue?

In [77]:
mycursor.execute("""
SELECT payment_method,
       ROUND(SUM(order_value), 2) AS total_revenue,
       COUNT(order_id) AS total_orders
FROM orders
GROUP BY payment_method
ORDER BY total_revenue DESC;
""")
out = mycursor.fetchall()
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))

+------------------+-----------------+----------------+
| payment_method   |   total_revenue |   total_orders |
|------------------+-----------------+----------------|
| Card             |     8.2722e+06  |           8364 |
| Cash             |     8.24543e+06 |           8384 |
| UPI              |     8.13442e+06 |           8252 |
+------------------+-----------------+----------------+
